# SPY Data — First Look EDA

Quick exploratory analysis of the processed SPY surface-point data.

**Prerequisites**: Run the pipeline steps first:
```bash
cd src/data
python 01_ingest_spy_github_dataset.py
python 02_inspect_spy_schema.py
python 03_build_spy_surface_table.py
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

# Adjust path if running from notebooks/ directory
import sys, pathlib
PROJECT_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src" / "data"))
from config import SURFACE_POINTS_FILE, SURFACE_POINTS_STRICT_FILE, PLOTS_DIR

In [ ]:
# Load processed data
df = pd.read_parquet(SURFACE_POINTS_FILE)
print(f"Conservative dataset: {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
df.head()

## 1. Row Counts by Year

In [ ]:
df["year"] = df["date"].dt.year
year_counts = df.groupby("year").size()

fig, ax = plt.subplots(figsize=(12, 4))
year_counts.plot.bar(ax=ax, color="steelblue")
ax.set_title("SPY Option Observations by Year")
ax.set_ylabel("Row count")
ax.set_xlabel("Year")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "spy_rows_by_year.png", dpi=150)
plt.show()

year_counts

## 2. Call vs Put Split

In [ ]:
type_counts = df["type"].str.lower().value_counts()
print(type_counts)
print(f"\nCall/Put ratio: {type_counts.get('call', 0) / max(type_counts.get('put', 1), 1):.2f}")

## 3. Implied Volatility Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["implied_volatility"], bins=200, color="steelblue", edgecolor="none")
axes[0].set_title("IV Distribution (full)")
axes[0].set_xlabel("Implied Volatility")
axes[0].set_xlim(0, 2)

# Zoom into typical range
mask = df["implied_volatility"].between(0.05, 1.0)
axes[1].hist(df.loc[mask, "implied_volatility"], bins=200, color="darkorange", edgecolor="none")
axes[1].set_title("IV Distribution (0.05–1.0)")
axes[1].set_xlabel("Implied Volatility")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "spy_iv_distribution.png", dpi=150)
plt.show()

df["implied_volatility"].describe()

## 4. Tau (Time to Expiry) Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["tau"], bins=200, color="teal", edgecolor="none")
ax.set_title("Tau Distribution (annualized)")
ax.set_xlabel("Tau (years)")
ax.set_xlim(0, 2.5)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "spy_tau_distribution.png", dpi=150)
plt.show()

df["tau"].describe()

## 5. Log-Moneyness Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["log_moneyness"], bins=300, color="purple", edgecolor="none")
ax.set_title("Log-Moneyness Distribution")
ax.set_xlabel("ln(K/S)")
ax.set_xlim(-1, 1)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "spy_log_moneyness_distribution.png", dpi=150)
plt.show()

df["log_moneyness"].describe()

## 6. IV Surface — Sample Dates

Scatter plots of tau vs log-moneyness colored by IV for a few sample dates.

In [ ]:
# Pick a few well-spaced sample dates
unique_dates = sorted(df["date"].unique())
n = len(unique_dates)
sample_indices = [n // 4, n // 2, 3 * n // 4, n - 1]
sample_dates = [unique_dates[i] for i in sample_indices]

# Use strict subset for cleaner plots
strict = pd.read_parquet(SURFACE_POINTS_STRICT_FILE)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, d in zip(axes.flat, sample_dates):
    sub = strict[strict["date"] == d]
    if len(sub) == 0:
        ax.set_title(f"{str(d)[:10]} — no data")
        continue
    sc = ax.scatter(
        sub["log_moneyness"], sub["tau"],
        c=sub["implied_volatility"], cmap="RdYlBu_r",
        s=8, alpha=0.6, vmin=0.05, vmax=0.8,
    )
    ax.set_xlabel("log(K/S)")
    ax.set_ylabel("τ (years)")
    ax.set_title(f"{str(d)[:10]} — {len(sub):,} points")
    ax.set_xlim(-0.5, 0.5)
    plt.colorbar(sc, ax=ax, label="IV")

plt.suptitle("SPY IV Surface — Sample Dates (strict subset)", fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "spy_iv_surface_samples.png", dpi=150)
plt.show()

## 7. Spread Diagnostics

In [ ]:
spread = (df["ask"] - df["bid"])
rel_spread = spread / np.maximum(df["mid"], 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(spread.clip(0, 5), bins=200, color="coral", edgecolor="none")
axes[0].set_title("Absolute Spread (clipped at $5)")
axes[0].set_xlabel("Ask - Bid ($)")

axes[1].hist(rel_spread.clip(0, 2), bins=200, color="coral", edgecolor="none")
axes[1].set_title("Relative Spread (clipped at 200%)")
axes[1].set_xlabel("(Ask - Bid) / Mid")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "spy_spread_diagnostics.png", dpi=150)
plt.show()

print("Flag summary:")
for col in ["flag_zero_bid", "flag_zero_volume", "flag_zero_oi", "flag_wide_spread"]:
    print(f"  {col}: {df[col].sum():,} ({df[col].mean()*100:.1f}%)")

## 8. Missingness Diagnostics

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False) * 100
null_pct = null_pct[null_pct > 0]

if len(null_pct) == 0:
    print("No missing values in the processed dataset.")
else:
    print("Columns with missing values (%):\n")
    print(null_pct.to_string())

## 9. Summary Stats — Strict Subset

In [ ]:
print(f"Conservative: {len(df):,} rows")
print(f"Strict:       {len(strict):,} rows ({len(strict)/len(df)*100:.1f}%)")
print(f"\nStrict date range: {strict['date'].min().date()} → {strict['date'].max().date()}")
print(f"Unique dates:      {strict['date'].nunique():,}")
print(f"\nStrict IV:  [{strict['implied_volatility'].min():.4f}, {strict['implied_volatility'].max():.4f}]")
print(f"Strict tau: [{strict['tau'].min():.4f}, {strict['tau'].max():.4f}]")
print(f"Strict lnK/S: [{strict['log_moneyness'].min():.4f}, {strict['log_moneyness'].max():.4f}]")